In [15]:
import asyncio
import time
def sync_task(name,seconds):
    print(f"  📤 {name} 开始（等 {seconds}s）")
    time.sleep(seconds)
    print(f"  📥 {name} 完成")
    return f"{name} 的结果"
async def async_task(name,seconds):
    print(f'{name}开始(等{seconds}s)')
    await asyncio.sleep(seconds)
    print(f'{name}完成')
    return f"{name} 的结果"
print("=== 同步（串行）===")
start = time.perf_counter()
sync_task('A',2)
sync_task('B',2)
sync_task('C',2)
print(f"耗时: {time.perf_counter()-start:.1f}s\n")

print("=== 异步（并发）===")
async def main():
    start = time.perf_counter()
    results = await asyncio.gather(
        async_task('A',2),
        async_task('B',2),
        async_task('C',2),
    )
    print(f"结果: {results}")
    print(f"耗时: {time.perf_counter()-start:.1f}s")
await main()

=== 同步（串行）===
  📤 A 开始（等 2s）
  📥 A 完成
  📤 B 开始（等 2s）
  📥 B 完成
  📤 C 开始（等 2s）
  📥 C 完成
耗时: 6.0s

=== 异步（并发）===
A开始(等2s)
B开始(等2s)
C开始(等2s)
A完成
B完成
C完成
结果: ['A 的结果', 'B 的结果', 'C 的结果']
耗时: 2.0s


In [22]:
import asyncio
import time
import random
async def fetch_embedding(text:str) -> list:
    await asyncio.sleep(random.uniform(0.2,0.8))
    return [random.gauss(0,1) for _ in range(768)]
async def main():
    texts = [f"文档片段 {i}" for i in range(10)]
    print("=== asyncio.TaskGroup (Python 3.11+) ===\n")
    start = time.perf_counter()
    results = []
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(fetch_embedding(t)) for t in texts]
    results = [task.result() for task in tasks]

    elapsed = time.perf_counter() - start
    print(f"✅ {len(results)} 个 Embedding 完成")
    print(f"⏱️  耗时: {elapsed:.2f}s")
    print(f"📐 向量维度: {len(results[0])}")

    print("\n--- 对比 ---")
    print("gather:")
    print("  - 一个失败 → 其他继续（return_exceptions=True）")
    print("  - 一个失败 → 默认取消其他（return_exceptions=False）")
    print("TaskGroup:")
    print("  - 一个失败 → 自动取消所有其他任务（结构化并发）")
    print("  - 异常会被包装成 ExceptionGroup 抛出")
    print("  - 更安全：不会'忘记'处理某个任务的异常")
await main()

=== asyncio.TaskGroup (Python 3.11+) ===

✅ 10 个 Embedding 完成
⏱️  耗时: 0.79s
📐 向量维度: 768

--- 对比 ---
gather:
  - 一个失败 → 其他继续（return_exceptions=True）
  - 一个失败 → 默认取消其他（return_exceptions=False）
TaskGroup:
  - 一个失败 → 自动取消所有其他任务（结构化并发）
  - 异常会被包装成 ExceptionGroup 抛出
  - 更安全：不会'忘记'处理某个任务的异常


In [35]:
import time
from concurrent.futures import ThreadPoolExecutor
def call_llm(prompt):
    time.sleep(2)
    return f"回答: {prompt}"
prompts = ["什么是RAG?", "什么是Agent?", "什么是Transformer?"]
start = time.time()
[call_llm(p) for p in prompts]
print(f"串行耗时: {time.time() - start:.1f}s")
start = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(call_llm,prompts))
print(f"并发耗时: {time.time() - start:.1f}s")

串行耗时: 6.0s
并发耗时: 2.0s


In [37]:
import time
from concurrent.futures import ThreadPoolExecutor
def call_llm(prompt):
    time.sleep(2)
    return f"回答: {prompt}"
prompts = ["什么是RAG?", "什么是Agent?", "什么是Transformer?"]
start = time.time()
[call_llm(p) for p in prompts]
print(time.time()-start)
start = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(call_llm,prompts))
print(time.time()-start)

6.005215644836426
2.0019166469573975


In [40]:
import time 
import random
from concurrent.futures import ThreadPoolExecutor
def get_embedding(text):
    time.sleep(random.uniform(0.2,0.5))
    return [random.random() for _ in range(768)]
docs =[f'这是第 {i} 个文档'for i in range(20)]
print(f"开始生成 {len(docs)} 个 Embedding...")
start = time.time()
with ThreadPoolExecutor(max_workers=10) as pool:
    vectors = list(pool.map(get_embedding,docs))
print(f"完成！耗时 {time.time() - start:.2f}s，生成了 {len(vectors)} 个向量")

开始生成 20 个 Embedding...
完成！耗时 0.84s，生成了 20 个向量
